In [0]:
from pyspark.sql import functions as F

bronze = spark.table("roadsafe.bronze.accidents")

def to_int(c): return F.col(c).cast("int")
def to_double(c): return F.regexp_replace(F.col(c), ",", ".").cast("double")

silver = (
    bronze
    .select(
        to_int("UJAHR").alias("year"),
        to_int("UMONAT").alias("month"),
        to_int("USTUNDE").alias("hour"),
        to_int("UWOCHENTAG").alias("weekday"),
        to_int("UKATEGORIE").alias("severity_code"),
        to_int("UART").alias("accident_kind_code"),
        to_int("ULICHTVERH").alias("light_code"),
        to_int("STRZUSTAND").alias("road_condition_code"),
        (to_int("IstPKW") == 1).alias("involves_cars"),
        (to_int("IstFuss") == 1).alias("involves_pedestrians"),
        (to_int("IstRad") == 1).alias("involves_bicycle"),
        (to_int("IstKrad") == 1).alias("involves_motorcycle"),
        (to_int("IstGkfz") == 1).alias("involves_truck"),
        to_double("LON").alias("longitude"),
        to_double("LAT").alias("lat"),
        "_source_file"
    )
)

In [0]:
silver = (
    silver
    .withColumn("severity",
                F.when(F.col("severity_code") == 1, "fatal")
                 .when(F.col("severity_code") == 2, "serious")
                 .when(F.col("severity_code") == 3, "slight"))
    .withColumn("light",
                F.when(F.col("light_code") == 0, "daylight")
                 .when(F.col("light_code") == 1, "twilight")
                 .when(F.col("light_code") == 2, "darkness"))
    .withColumn("road_condition",
                F.when(F.col("road_condition_code") == 0, "dry")
                .when(F.col("road_condition_code") == 1, "wet")
                .when(F.col("road_condition_code") == 2, "snow"))
)

## Filter garbage. Germany's rough bounding box is latitude 47 to 55.5 and longitude 5.5 to 15.5

In [0]:
valid_coords = (F.col("lat").between(47.0, 55.0) & F.col("longitude").between(5.5, 15.5))
clean = silver.filter(valid_coords & F.col("severity_code").isin(1, 2, 3))
rejected = silver.filter(~(valid_coords & F.col("severity_code").isin(1, 2, 3)))

## Drop exact duplicates and save

In [0]:
clean = clean.dropDuplicates()
clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("roadsafe.silver.accidents")
rejected.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("roadsafe.silver.accidents_rejected")

## Compare the counts, which is a vital sanity check:

In [0]:
%sql
SELECT
(SELECT COUNT(*) FROM roadsafe.bronze.accidents) AS bronze_rows,
(SELECT COUNT(*) FROM roadsafe.silver.accidents) AS silver_rows,
(SELECT COUNT(*) FROM roadsafe.silver.accidents_rejected) AS rejected_rows;